<a href="https://colab.research.google.com/github/OpenConceptLab/ocl-content/blob/main/OCL_Hierarchy_Amend_File_Maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is designed to process OCL ICD-11 concepts data. It loads concept data from a JSON file, processes it to extract parent-child relationships, and then generates a `hierarchy_amend.json` file that defines these relationships. This `hierarchy_amend.json` can then be used to update a hierarchy in a system like OpenConceptLab (OCL). This hierarchy_amend.json can be used via OCL's Amend Hierarchy i.e. `POST /admin/concepts/amend-hierarchy/`.

To use this, adjust the variables in the Config cell to match your owner and source, plus the Input and Output file paths.

In [ ]:
CONFIG = {
    "settings": {
        "parent_url_template": "/orgs/{owner}/sources/{source}/concepts/{concept_id}/"
    },
    "constants": {
        "owner": "OpenMRS-OCL-Squad", # Assuming from filename
        "source": "ICD-11-WHO", # Assuming from filename
    },
    "hierarchy_amend_output_path": "C:\\Users\\jamlung\\Documents\\OCL-CIEL-Madiro 2026\\ICD-11-WHO-hierarchy_amend.json"
}

input_file_path = "C:\\Users\\jamlung\\Documents\\OCL-CIEL-Madiro 2026\\ocl_icd11.json"

In [ ]:
import json

processed_concepts_data = []

with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        processed_concepts_data.append(json.loads(line))

print(f"Loaded {len(processed_concepts_data)} concepts from {input_file_path}")

In [ ]:
hierarchy_amend_data = {}

for concept in processed_concepts_data:
    concept_id = concept['id']
    # Construct the child's own URL
    child_url = CONFIG['settings']['parent_url_template'].format(
        owner=CONFIG['constants']['owner'],
        source=CONFIG['constants']['source'],
        concept_id=concept_id
    )

    # Iterate through parents and add child to the parent's list
    for parent_url in concept.get('parent_concept_urls', []):
        if parent_url not in hierarchy_amend_data:
            hierarchy_amend_data[parent_url] = []
        hierarchy_amend_data[parent_url].append(child_url)

# Define the output path for the hierarchy amend file
output_amend_path = CONFIG['hierarchy_amend_output_path']

# Write the hierarchy_amend_data to a JSON file
print(f"Writing hierarchy data to {output_amend_path}...")
with open(output_amend_path, "w") as f:
    json.dump(hierarchy_amend_data, f, indent=2)

print("Hierarchy amend file created successfully!")

In [ ]:
with open(output_amend_path, "r") as f:
    generated_amend_data = json.load(f)

display(generated_amend_data)